In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoTokenizer

In [2]:
def truncated_normal_(tensor, std=1.0, trunc=2.0):
    size = tensor.shape
    tmp = tensor.new_empty(size + (4,)).normal_()
    valid = (tmp < trunc) & (tmp > -trunc)
    ind = valid.max(-1, keepdim=True)[1]
    tensor.data.copy_(tmp.gather(-1, ind).squeeze(-1))
    tensor.data.mul_(std)
    return tensor

class RotaryPositionalEncoding(nn.Module):
    def __init__(self, dim, max_seq_len=2048):
        super().__init__()
        self.dim = dim
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(0, max_seq_len, dtype=torch.float)
        sinusoid_inp = torch.ger(positions, inv_freq)
        self.register_buffer("sin", torch.sin(sinusoid_inp))
        self.register_buffer("cos", torch.cos(sinusoid_inp))

    def forward(self, x):
        seq_len = x.size(1)
        sin = self.sin[:seq_len, :].unsqueeze(0)
        cos = self.cos[:seq_len, :].unsqueeze(0)
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        x[..., ::2] = x1 * cos - x2 * sin
        x[..., 1::2] = x1 * sin + x2 * cos
        return x

class GLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim * 2, bias=False)

    def forward(self, x):
        x_proj = self.proj(x)
        a, b = x_proj.chunk(2, dim=-1)
        return a * torch.sigmoid(b)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.dim = dim

    def forward(self, x):
        norm = x.norm(dim=-1, keepdim=True) * math.sqrt(self.dim)
        return x / (norm + self.eps)

class EncoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, bias=False, batch_first=True)
        self.rmsnorm1 = RMSNorm(dim)
        self.mlp = nn.Sequential(
            GLU(dim),
            nn.Linear(dim, dim, bias=False)
        )
        self.rmsnorm2 = RMSNorm(dim)
        self.rotary_pe = RotaryPositionalEncoding(dim)

    def forward(self, x):
        x = self.rotary_pe(x)
        attn_out, _ = self.attn(x, x, x, need_weights=False)
        x = x + attn_out
        x = self.rmsnorm1(x)
        mlp_out = self.mlp(x)
        x = x + mlp_out
        x = self.rmsnorm2(x)
        return x

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim)

    def forward(self, x):
        return self.embedding(x)

class OutputHead(nn.Module):
    def __init__(self, dim, vocab_size, stablemax=False):
        super().__init__()
        self.linear = nn.Linear(dim, vocab_size, bias=False)
        self.stablemax = stablemax

    def forward(self, x):
        logits = self.linear(x)
        if self.stablemax:
            logits = logits - logits.max(dim=-1, keepdim=True)[0]
            exp_logits = torch.exp(logits)
            probs = exp_logits / exp_logits.sum(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits, dim=-1)
        return probs

class HRMSeq2Seq(nn.Module):
    def __init__(self, vocab_size, dim, num_layers=2, num_heads=4, stablemax=False):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, dim)
        self.high_level = nn.Sequential(
            *[EncoderBlock(dim, num_heads) for _ in range(num_layers)]
        )
        self.low_level = nn.Sequential(
            *[EncoderBlock(dim, num_heads) for _ in range(num_layers)]
        )
        self.output_head = OutputHead(dim, vocab_size, stablemax=stablemax)
        self.z0 = nn.Parameter(torch.zeros(1, dim), requires_grad=False)
        truncated_normal_(self.z0, std=1.0, trunc=2.0)

    def forward(self, src):
        x = self.embedding(src)
        x = x + self.z0
        x = self.high_level(x)
        x = self.low_level(x)
        probs = self.output_head(x)
        return probs

In [3]:
Tokenizer_name = "gpt2"
max_prompt_len = 100
dim = 64
num_layers = 2
num_heads = 4
stablemax = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
tokenizer = AutoTokenizer.from_pretrained(Tokenizer_name)
tokenizer.pad_token = tokenizer.eos_token
vocab_size = tokenizer.vocab_size

In [5]:
model = HRMSeq2Seq(vocab_size, dim, num_layers, num_heads, stablemax).to(device)
model.load_state_dict(torch.load("hrmseq2seq.pt", map_location=device))
model.eval()

HRMSeq2Seq(
  (embedding): TokenEmbedding(
    (embedding): Embedding(50257, 64)
  )
  (high_level): Sequential(
    (0): EncoderBlock(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=False)
      )
      (rmsnorm1): RMSNorm()
      (mlp): Sequential(
        (0): GLU(
          (proj): Linear(in_features=64, out_features=128, bias=False)
        )
        (1): Linear(in_features=64, out_features=64, bias=False)
      )
      (rmsnorm2): RMSNorm()
      (rotary_pe): RotaryPositionalEncoding()
    )
    (1): EncoderBlock(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=False)
      )
      (rmsnorm1): RMSNorm()
      (mlp): Sequential(
        (0): GLU(
          (proj): Linear(in_features=64, out_features=128, bias=False)
        )
        (1): Linear(in_features=64, out_features=64, bias=False)
      )
      (rmsnorm2): RMSNorm()
     

In [6]:
prompt = "hello world"
prompt_ids = tokenizer(prompt, padding="max_length", truncation=True, max_length=max_prompt_len, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    probs = model(prompt_ids)  # shape: [1, max_prompt_len, vocab_size]
    predicted_ids = probs.argmax(dim=-1)    # shape: [1, max_prompt_len]

In [7]:
# Decode (note: output is max_prompt_len tokens, not a full response sequence)
output_text = tokenizer.decode(predicted_ids[0].cpu().numpy(), skip_special_tokens=True)
print("Model output:", output_text)

Model output: Here some some sample implementation of which functionality in in nose:
 for engineers`


 for
                      
 

  task
 a ` : 

:
:::::::

 

 



 a a a and  

 a and

 and





:



 a the a
::
